# LeetCode 538 — Convert BST to Greater Tree bằng Python

Link đề: https://leetcode.com/problems/convert-bst-to-greater-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Search Tree, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def convertBST(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        ...
```

Ghi chú: notebook tự dựng `TreeNode`, hàm `build_tree` (dựng cây từ list kiểu LeetCode như `[4,1,6,0,2,5,7,null,null,null,3,null,null,null,8]`) và `tree_to_list` (xuất cây ra list cùng kiểu). Bài này giống hệt bài **1038. Binary Search Tree to Greater Sum Tree**. Ví dụ cây rỗng `[]` là do notebook tự đặt (ràng buộc đề cho phép 0 nút), đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho một **cây tìm kiếm nhị phân (BST)**, đổi giá trị của **mỗi nút** thành **chính nó + tổng mọi giá trị lớn hơn nó** trong cây (tức là tổng mọi giá trị **lớn hơn hoặc bằng** nó, tính theo *giá trị gốc ban đầu*).
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, dựng/xuất cây từ list có `None`, đệ quy với biến chia sẻ (`nonlocal`, `self.total`), danh sách `list` làm **ngăn xếp (stack)**, `itertools.accumulate`, và cách **sửa cây tại chỗ (in-place)**.
3. Đi từ **với mỗi nút, quét cả cây để cộng các khóa lớn hơn** → **duyệt trung thứ tự ra list rồi tính tổng hậu tố** → **duyệt trung thứ tự đảo (phải – gốc – trái) với biến tổng chạy** → **bản lặp dùng stack** → **Morris traversal dùng O(1) bộ nhớ phụ**.
4. Hiểu mấu chốt: duyệt trung thứ tự (trái – gốc – phải) một BST cho dãy **tăng dần**; duyệt **đảo** (phải – gốc – trái) cho dãy **giảm dần**, nên khi tới một nút, biến tổng chạy **đã chứa đúng mọi khóa lớn hơn nó**.
5. Tránh các lỗi kinh điển: truyền tổng theo kiểu **tham trị** nên mất khi quay lui, duyệt sai chiều (ra tổng các khóa *nhỏ hơn*), sửa giá trị ngay trong lúc brute force, biến đếm ở **mức class** bị dính giữa các lần gọi, và đệ quy quá sâu với cây lệch.

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một **cây tìm kiếm nhị phân (BST)**. Hãy chuyển nó thành **Greater Tree** sao cho **mỗi khóa (key) của BST gốc** được đổi thành **khóa gốc cộng với tổng tất cả các khóa lớn hơn nó** trong BST.

Nhắc lại: BST thỏa các ràng buộc sau:

- Cây con **trái** của một nút chỉ chứa khóa **nhỏ hơn** khóa của nút.
- Cây con **phải** của một nút chỉ chứa khóa **lớn hơn** khóa của nút.
- Cả hai cây con cũng là BST.

**Ví dụ 1**

- Input: `root = [4,1,6,0,2,5,7,null,null,null,3,null,null,null,8]`
- Output: `[30,36,21,36,35,26,15,null,null,null,33,null,null,null,8]`
- Giải thích: các khóa gốc là `0..8`, mỗi khóa đổi thành tổng của nó và mọi khóa lớn hơn: `8 -> 8`, `7 -> 7+8 = 15`, `6 -> 21`, `5 -> 26`, `4 -> 30`, `3 -> 33`, `2 -> 35`, `1 -> 36`, `0 -> 36`.

**Ví dụ 2**

- Input: `root = [0,null,1]`
- Output: `[1,null,1]`

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = []`
- Output: `[]`

**Ràng buộc:**

- Số nút của cây nằm trong khoảng `[0, 10^4]` (**có thể bằng 0**).
- `-10^4 <= Node.val <= 10^4`.
- Mọi giá trị trong cây là **duy nhất**.
- `root` được đảm bảo là một BST hợp lệ.

Chú ý: hàm **trả về gốc của cây đã biến đổi** (thường là sửa tại chỗ rồi trả lại chính `root`). Giá trị có thể **âm**, nên "tổng các khóa lớn hơn" có thể là số âm.

In [1]:
from typing import Optional, List
from collections import deque
from itertools import accumulate
import random
import sys
import time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([4, 1, 6, 0, 2, 5, 7, None, None, None, 3, None, None, None, 8],
     [30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8]),
    ([0, None, 1], [1, None, 1]),
    ([], []),
]
for vals, expected in examples:
    assert tree_to_list(build_tree(vals)) == vals      # dựng rồi xuất phải ra lại đúng list ban đầu
    print(vals, "-> kỳ vọng", expected)

[4, 1, 6, 0, 2, 5, 7, None, None, None, 3, None, None, None, 8] -> kỳ vọng [30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8]
[0, None, 1] -> kỳ vọng [1, None, 1]
[] -> kỳ vọng []


## 3. Kiến thức nền

### 3.1. `TreeNode`, dựng cây và xuất cây

Mỗi nút của cây nhị phân có `val`, `left`, `right` (con trái, con phải, có thể là `None` nghĩa là "không có"). Trên LeetCode, cây được cho dưới dạng **list theo từng tầng, từ trái sang phải**, `None` đánh dấu chỗ trống. Hai hàm ở cell trên:

- `build_tree(values)`: dùng một hàng đợi `deque`; lấy nút đang chờ ra, gán con trái rồi con phải từ list, rồi đưa các con thật vào hàng đợi.
- `tree_to_list(root)`: làm ngược lại và cắt các `None` thừa ở cuối. Ta dùng nó để **kiểm tra kết quả** của mọi lời giải.

Hãy thử dựng cây ví dụ 1 và nhìn các con của nó.

In [2]:
t = build_tree(examples[0][0])
print("gốc:", t.val, "| con trái:", t.left.val, "| con phải:", t.right.val)
print("con của 1:", t.left.left.val, t.left.right.val, "| con của 6:", t.right.left.val, t.right.right.val)
print("nút 2 có con phải:", t.left.right.right.val, "| nút 2 có con trái:", t.left.right.left)
print(tree_to_list(t))
print("cây rỗng:", build_tree([]), tree_to_list(None))

gốc: 4 | con trái: 1 | con phải: 6
con của 1: 0 2 | con của 6: 5 7
nút 2 có con phải: 3 | nút 2 có con trái: None
[4, 1, 6, 0, 2, 5, 7, None, None, None, 3, None, None, None, 8]
cây rỗng: None []


### 3.2. BST và duyệt trung thứ tự (inorder)

BST có tính chất: **trái < gốc < phải** ở mọi nút. Hệ quả cực quan trọng: duyệt **trung thứ tự** (inorder = *trái → gốc → phải*) một BST sẽ gặp các khóa theo thứ tự **tăng dần**. Lý do: mọi khóa ở cây con trái đều nhỏ hơn gốc nên được thăm trước gốc; mọi khóa ở cây con phải đều lớn hơn nên được thăm sau; áp dụng đệ quy cho từng cây con.

In [3]:
def inorder(root):
    """Trả về list khóa theo thứ tự trái -> gốc -> phải."""
    if root is None:
        return []
    return inorder(root.left) + [root.val] + inorder(root.right)

tree = build_tree(examples[0][0])
print(inorder(tree))                       # tăng dần: 0..8
print(inorder(tree) == sorted(inorder(tree)))

[0, 1, 2, 3, 4, 5, 6, 7, 8]
True


### 3.3. Duyệt trung thứ tự **đảo** (reverse inorder): phải → gốc → trái

Nếu đổi thứ tự thành **phải → gốc → trái**, ta gặp các khóa theo thứ tự **giảm dần**. Đây là chìa khóa của bài này: khi ta đứng ở một nút, **mọi nút đã được thăm trước đó đều có khóa lớn hơn nó**, và ngược lại mọi nút có khóa lớn hơn nó **đều đã được thăm**.

In [4]:
def reverse_inorder(root):
    if root is None:
        return []
    return reverse_inorder(root.right) + [root.val] + reverse_inorder(root.left)

print(reverse_inorder(build_tree(examples[0][0])))       # giảm dần: 8..0

[8, 7, 6, 5, 4, 3, 2, 1, 0]


### 3.4. Biến chia sẻ trong đệ quy: `nonlocal` và `self.total`

Ta muốn một biến `total` "sống" xuyên suốt mọi lời gọi đệ quy. Python có hai cách quen thuộc:

- Hàm lồng nhau + từ khóa **`nonlocal total`**: cho phép hàm con **gán lại** biến của hàm cha (không có `nonlocal`, phép gán `total += ...` sẽ tạo biến cục bộ mới và báo `UnboundLocalError`).
- Thuộc tính của đối tượng: `self.total`.

Số nguyên trong Python là **bất biến** và được **truyền theo kiểu tham trị**: nếu viết `def dfs(node, s)` rồi `s += node.val` bên trong thì `s` ở lời gọi cha **không đổi**. Đó là nguồn của một lỗi kinh điển ở mục 13.

In [5]:
def counter_demo():
    count = 0
    def bump():
        nonlocal count          # bỏ dòng này sẽ báo UnboundLocalError
        count += 1
    for _ in range(3):
        bump()
    return count

print(counter_demo())

def by_value(s):
    s += 100                    # chỉ đổi biến cục bộ s
    return s

x = 5
by_value(x)
print("x vẫn là", x)

3
x vẫn là 5


### 3.5. Tổng hậu tố (suffix sum) và `itertools.accumulate`

Với dãy `a = [a0, a1, ..., a(n-1)]`, **tổng hậu tố** tại vị trí `i` là `a[i] + a[i+1] + ... + a[n-1]`. Cách tính nhanh: đi **từ phải sang trái**, giữ một tổng chạy. `itertools.accumulate(it)` trả về các **tổng tích lũy** (tổng tiền tố) của một dãy; để có tổng hậu tố, ta đảo dãy, tích lũy, rồi đảo lại.

In [6]:
a = [0, 1, 2, 3, 4]
print(list(accumulate(a)))                           # tổng tiền tố: 0, 1, 3, 6, 10
suffix = list(accumulate(reversed(a)))[::-1]        # tổng hậu tố
print(suffix)                                        # [10, 10, 9, 7, 4]

# viết tay: đi từ phải sang trái với tổng chạy
run, out = 0, [0] * len(a)
for i in range(len(a) - 1, -1, -1):
    run += a[i]
    out[i] = run
print(out)

[0, 1, 3, 6, 10]
[10, 10, 9, 7, 4]
[10, 10, 9, 7, 4]


### 3.6. `list` làm ngăn xếp (stack) cho duyệt không đệ quy

`list.append(x)` đẩy vào cuối, `list.pop()` lấy ra từ cuối — vào sau ra trước (LIFO). Ta dùng stack để **tự mô phỏng** cuộc gọi đệ quy, tránh lỗi `RecursionError` với cây rất lệch.

### 3.7. Sửa cây tại chỗ và vì sao test phải dựng cây mới

Lời giải của ta **sửa `node.val` trực tiếp** rồi trả lại `root`. Hệ quả: sau khi chạy, cây gốc đã bị đổi. Trong test, muốn chạy nhiều cách trên **cùng một input** thì phải **dựng cây mới** từ list cho mỗi lần (`build_tree(vals)`), nếu không cách thứ hai sẽ nhận cây đã bị cách thứ nhất biến đổi.

In [7]:
vals = [0, None, 1]
r = build_tree(vals)
print("trước:", tree_to_list(r))
r.val += r.right.val                 # sửa tại chỗ
print("sau  :", tree_to_list(r))      # cây đã đổi
print("dựng lại từ list gốc:", tree_to_list(build_tree(vals)))

trước: [0, None, 1]
sau  : [1, None, 1]
dựng lại từ list gốc: [0, None, 1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy bỏ hình dạng cây đi, chỉ nhìn các **khóa**. Ví dụ 1 có các khóa `0,1,2,3,4,5,6,7,8` (khi sắp tăng dần). Với mỗi khóa `k`, đáp án là **`k` cộng mọi khóa lớn hơn `k`**:

| khóa `k` | các khóa lớn hơn | đáp án mới |
|---|---|---|
| 8 | (không có) | 8 |
| 7 | 8 | 7 + 8 = 15 |
| 6 | 7, 8 | 6 + 15 = 21 |
| 5 | 6, 7, 8 | 5 + 21 = 26 |
| 4 | 5..8 | 4 + 26 = 30 |
| 3 | 4..8 | 3 + 30 = 33 |
| 2 | 3..8 | 2 + 33 = 35 |
| 1 | 2..8 | 1 + 35 = 36 |
| 0 | 1..8 | 0 + 36 = 36 |

Để ý: đáp án của `k` = **`k` + đáp án của khóa liền trên nó**. Nghĩa là nếu ta đi **từ khóa lớn nhất xuống khóa nhỏ nhất** và giữ một **tổng chạy**, mỗi khóa chỉ cần cộng thêm vào tổng đó.

Tìm lại đúng thứ tự "lớn nhất → nhỏ nhất" trong cây? Chính là duyệt **phải → gốc → trái** (mục 3.3). Vậy bài toán hạ xuống: *duyệt đảo trung thứ tự một lần, giữ tổng chạy, ghi tổng đó vào từng nút*.

Cell dưới kiểm chứng phép tính tay bằng code: ghép các khóa với đáp án rồi so với đầu ra mẫu.

In [8]:
keys = sorted(inorder(build_tree(examples[0][0])))
new_of = {}
run = 0
for k in reversed(keys):
    run += k
    new_of[k] = run
print(new_of)

# đối chiếu với đầu ra mẫu: map từng khóa trong list đầu vào sang giá trị mới
mapped = [None if v is None else new_of[v] for v in examples[0][0]]
print(mapped)
assert mapped == examples[0][1]

{8: 8, 7: 15, 6: 21, 5: 26, 4: 30, 3: 33, 2: 35, 1: 36, 0: 36}
[30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8]


## 5. Cách 1 — Brute Force / baseline: với mỗi nút, quét cả cây để cộng các khóa lớn hơn

**Ý tưởng.** Làm đúng như đề phát biểu:

1. Duyệt cây một lần, gom **tất cả nút** vào một list `nodes`.
2. Với mỗi nút `x` trong `nodes`, tính `new_val = tổng các m.val với m.val >= x.val` bằng cách **quét hết** `nodes`.
3. **Chỉ sau khi tính xong cho mọi nút**, mới gán giá trị mới (nếu gán ngay trong lúc quét thì các tổng sau sẽ dùng nhầm giá trị đã đổi — xem lỗi ở mục 13).

Cách này không dùng tới tính chất BST nào cả: nó đúng cho mọi cây có khóa duy nhất. Hãy chạy thử trên các ví dụ.

In [9]:
def collect_nodes(root):
    """Gom mọi nút của cây (preorder), dùng ngăn xếp để không đệ quy."""
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        out.append(node)
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
    return out


def convert_brute(root):
    nodes = collect_nodes(root)
    # BƯỚC 1: tính hết giá trị mới (dựa trên giá trị GỐC)
    new_vals = [sum(m.val for m in nodes if m.val >= x.val) for x in nodes]
    # BƯỚC 2: gán sau cùng
    for x, v in zip(nodes, new_vals):
        x.val = v
    return root


for vals, expected in examples:
    got = tree_to_list(convert_brute(build_tree(vals)))
    print(got)
    assert got == expected

[30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8]
[1, None, 1]
[]


**Độ phức tạp.** Với `n` nút, mỗi nút quét `n` nút nên thời gian **O(n²)**; bộ nhớ **O(n)** cho list `nodes` và `new_vals`. Với `n = 10^4` là khoảng `10^8` phép so sánh/cộng — rất chậm trong Python.

**Nút thắt.** Ta tính lại gần như cùng một tổng cho từng nút: tổng các khóa `>= x` và tổng các khóa `>= y` (với `y` liền kề) khác nhau đúng một số hạng. Ta chưa dùng *thứ tự* mà BST cho sẵn. Hãy khai thác thứ tự đó.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát ghép lại:

1. **Đáp án của một khóa = chính nó + đáp án của khóa liền trên** (mục 4) — tức là chỉ cần một **tổng chạy** khi đi từ lớn đến nhỏ.
2. BST cho ta thứ tự lớn → nhỏ **miễn phí** bằng duyệt **phải → gốc → trái**.

Vậy thuật toán là:

```
total = 0
duyệt đảo (phải -> gốc -> trái):
    tới nút x:  total += x.val ;  x.val = total
```

Lý do an toàn khi sửa `x.val` ngay tại chỗ: nút `x` chỉ được **đọc một lần duy nhất** (để cộng vào `total`) **trước khi** bị ghi đè, và không nút nào được thăm sau đó cần đọc lại giá trị gốc của `x`. Mỗi nút được thăm đúng một lần nên thời gian **O(n)**.

Phần còn lại của notebook là cùng một ý tưởng ở các "hình thức" khác nhau: tách thành hai pha qua list (cách 2), đệ quy (cách 3), ngăn xếp (cách 4), và Morris để bỏ cả ngăn xếp (cách 5).

## 7. Các cách tiếp theo

Tổng cộng ta có 5 cách: cách 1 (brute force, mục 5) và bốn cách dưới đây.

### 7.1. Cách 2 — Duyệt trung thứ tự ra list, tính tổng hậu tố, gán lại (hai pha)

**Giải thích từng đoạn.**

- *Pha 1:* duyệt **trung thứ tự** để lấy các **nút** (không phải chỉ giá trị) theo thứ tự tăng dần vào list `order`.
- *Pha 2:* đi `order` từ **phải sang trái** với tổng chạy `run`; ở mỗi nút `run += node.val` rồi `node.val = run`.

Cách này dễ hiểu nhất vì tách "đọc" và "ghi" thành hai pha riêng, nhưng cần bộ nhớ phụ `O(n)` cho `order`.

In [10]:
def convert_two_pass(root, verbose=False):
    order = []                                   # các nút theo thứ tự tăng dần

    def inorder_nodes(node):
        if node is None:
            return
        inorder_nodes(node.left)
        order.append(node)
        inorder_nodes(node.right)

    inorder_nodes(root)
    if verbose:
        print("khóa theo inorder:", [n.val for n in order])

    run = 0
    for node in reversed(order):                 # từ khóa lớn nhất xuống nhỏ nhất
        run += node.val
        node.val = run
    if verbose:
        print("sau pha 2        :", [n.val for n in order])
    return root


for vals, expected in examples:
    got = tree_to_list(convert_two_pass(build_tree(vals), verbose=bool(vals)))
    assert got == expected
print("OK")

khóa theo inorder: [0, 1, 2, 3, 4, 5, 6, 7, 8]
sau pha 2        : [36, 36, 35, 33, 30, 26, 21, 15, 8]
khóa theo inorder: [0, 1]
sau pha 2        : [1, 1]
OK


**Code nộp LeetCode (cách 2):**

In [11]:
class Solution2:
    def convertBST(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        order = []

        def inorder_nodes(node):
            if node is None:
                return
            inorder_nodes(node.left)
            order.append(node)
            inorder_nodes(node.right)

        inorder_nodes(root)
        run = 0
        for node in reversed(order):
            run += node.val
            node.val = run
        return root


for vals, expected in examples:
    assert tree_to_list(Solution2().convertBST(build_tree(vals))) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp cách 2.** Thời gian **O(n)** (hai lượt, mỗi lượt `n` nút). Bộ nhớ **O(n)** cho list `order`, cộng ngăn xếp đệ quy `O(h)` (`h` là chiều cao cây, tối đa `n`).

### 7.2. Cách 3 — Duyệt trung thứ tự đảo bằng đệ quy với tổng chạy (một pha)

**Giải thích từng đoạn.**

- Hàm `dfs(node)`: nếu `node is None` thì dừng.
- Gọi `dfs(node.right)` **trước**: xử lý hết mọi khóa lớn hơn `node` (cả cây phải).
- Tới `node`: `total += node.val`, rồi `node.val = total`.
- Gọi `dfs(node.left)`: các khóa nhỏ hơn, lúc này `total` đã gồm cả `node` và toàn bộ cây phải.

`total` phải là biến **dùng chung** (`nonlocal`), không phải tham số truyền theo trị.

In [12]:
def convert_recursive(root):
    total = 0

    def dfs(node):
        nonlocal total
        if node is None:
            return
        dfs(node.right)          # 1) mọi khóa lớn hơn
        total += node.val        # 2) cộng nút hiện tại
        node.val = total         #    ghi đè bằng tổng chạy
        dfs(node.left)           # 3) mọi khóa nhỏ hơn

    dfs(root)
    return root


for vals, expected in examples:
    got = tree_to_list(convert_recursive(build_tree(vals)))
    assert got == expected
print("OK")

OK


**Code nộp LeetCode (cách 3):**

In [13]:
class Solution3:
    def convertBST(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        total = 0

        def dfs(node: Optional[TreeNode]) -> None:
            nonlocal total
            if node is None:
                return
            dfs(node.right)
            total += node.val
            node.val = total
            dfs(node.left)

        dfs(root)
        return root


for vals, expected in examples:
    assert tree_to_list(Solution3().convertBST(build_tree(vals))) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp cách 3.** Thời gian **O(n)**: mỗi nút được thăm đúng một lần. Bộ nhớ **O(h)** cho ngăn xếp đệ quy: `O(log n)` với cây cân bằng, nhưng **`O(n)`** với cây lệch (xem lỗi `RecursionError` ở mục 13).

### 7.3. Cách 4 — Phiên bản lặp dùng ngăn xếp tự quản

**Giải thích từng đoạn.** Ta mô phỏng đệ quy "phải → gốc → trái" bằng một `stack`:

1. **Đi sâu hết bên phải:** từ `node` hiện tại, đẩy `node` vào `stack` rồi `node = node.right` cho tới khi hết.
2. **Pop một nút** ra: đây là nút lớn nhất chưa xử lý. Làm `total += node.val; node.val = total`.
3. **Rẽ sang trái:** `node = node.left` và quay lại bước 1 để đi hết nhánh phải của cây con trái.
4. Lặp cho tới khi `stack` rỗng **và** `node` là `None`.

In [14]:
def convert_stack(root):
    stack, node, total = [], root, 0
    while stack or node:
        while node:                  # 1) đi sâu về bên phải
            stack.append(node)
            node = node.right
        node = stack.pop()           # 2) nút lớn nhất còn lại
        total += node.val
        node.val = total
        node = node.left             # 3) rẽ trái, vòng lặp ngoài lặp lại bước 1
    return root


for vals, expected in examples:
    got = tree_to_list(convert_stack(build_tree(vals)))
    assert got == expected
print("OK")

OK


**Code nộp LeetCode (cách 4):**

In [15]:
class Solution4:
    def convertBST(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        stack: List[TreeNode] = []
        node, total = root, 0
        while stack or node:
            while node:
                stack.append(node)
                node = node.right
            node = stack.pop()
            total += node.val
            node.val = total
            node = node.left
        return root


for vals, expected in examples:
    assert tree_to_list(Solution4().convertBST(build_tree(vals))) == expected
print("Solution4 OK")

Solution4 OK


**Độ phức tạp cách 4.** Thời gian **O(n)** (mỗi nút được push một lần và pop một lần). Bộ nhớ **O(h)** cho `stack` (tối đa `n` với cây lệch) — nhưng đây là bộ nhớ *heap*, không dính giới hạn đệ quy của Python.

### 7.4. Cách 5 — Morris traversal (đảo): O(1) bộ nhớ phụ

**Vấn đề.** Cách 3 và 4 tốn `O(h)` bộ nhớ vì cần "nhớ đường quay lên". **Morris traversal** mượn tạm các con trỏ `None` của cây để làm "đường quay lại", nên không cần stack.

**Ý tưởng (phiên bản đảo, gốc → phải trước).** Tại nút `node`:

- Nếu `node.right is None`: không có gì lớn hơn để chờ, **xử lý `node` ngay** (`total += node.val; node.val = total`) rồi `node = node.left`.
- Ngược lại, tìm **`succ`** = nút **nhỏ nhất** của cây con phải (đi sang trái liên tục từ `node.right`). Đó chính là nút được thăm **ngay trước** `node` trong thứ tự đảo.
  - Nếu `succ.left is None`: lần đầu tới đây. Đặt **dây tạm** `succ.left = node` (để sau khi duyệt xong cây phải, từ `succ` ta nhảy ngược về `node`), rồi `node = node.right`.
  - Nếu `succ.left is node`: ta đã duyệt xong cây phải và vừa nhảy ngược về. **Gỡ dây** `succ.left = None`, **xử lý `node`**, rồi `node = node.left`.

Khi kết thúc, mọi dây tạm đã được gỡ nên cây khôi phục đúng cấu trúc, chỉ khác giá trị.

In [16]:
def convert_morris(root):
    total, node = 0, root
    while node:
        if node.right is None:
            total += node.val
            node.val = total
            node = node.left
        else:
            succ = node.right
            while succ.left and succ.left is not node:   # nhỏ nhất của cây phải
                succ = succ.left
            if succ.left is None:          # lần đầu: nối dây tạm rồi đi vào cây phải
                succ.left = node
                node = node.right
            else:                          # lần hai: đã duyệt xong cây phải
                succ.left = None           # gỡ dây
                total += node.val
                node.val = total
                node = node.left
    return root


for vals, expected in examples:
    got = tree_to_list(convert_morris(build_tree(vals)))
    assert got == expected
print("OK")

OK


**Code nộp LeetCode (cách 5):**

In [17]:
class Solution5:
    def convertBST(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        total, node = 0, root
        while node:
            if node.right is None:
                total += node.val
                node.val = total
                node = node.left
            else:
                succ = node.right
                while succ.left and succ.left is not node:
                    succ = succ.left
                if succ.left is None:
                    succ.left = node
                    node = node.right
                else:
                    succ.left = None
                    total += node.val
                    node.val = total
                    node = node.left
        return root


for vals, expected in examples:
    assert tree_to_list(Solution5().convertBST(build_tree(vals))) == expected
print("Solution5 OK")

Solution5 OK


**Độ phức tạp cách 5.** Thời gian **O(n)**: mỗi cạnh của cây bị đi qua tối đa hai lần (một lần để nối dây, một lần để gỡ), nên tổng công việc vẫn tuyến tính dù có vòng `while succ.left ...`. Bộ nhớ phụ **O(1)**. Đổi lại code khó đọc và **tạm thời làm cây không còn là cây** trong lúc chạy (nên không an toàn nếu có luồng khác đọc cây cùng lúc).

## 8. TRACE từng bước

Ta trace trên cây nhỏ `[5,2,13]` (khóa `2 < 5 < 13`; kỳ vọng `[18,20,13]`), vì cây ví dụ 1 có 9 nút nên bảng quá dài; cuối mục có thêm trace cây ví dụ 1 ở dạng gọn.

### 8.1. Trace cách 3 (đệ quy phải → gốc → trái)

Bảng trạng thái:

| Cột | Ý nghĩa |
|---|---|
| thụt lề | độ sâu của lời gọi đệ quy |
| `vào` / `ra` | đi vào nút / thăm nút (cộng và ghi đè) |
| `total` | tổng chạy sau khi xử lý nút |

In [18]:
def trace_recursive(root):
    total = 0

    def dfs(node, depth):
        nonlocal total
        pad = "    " * depth
        if node is None:
            print(f"{pad}(None) -> quay lại")
            return
        print(f"{pad}vào nút {node.val}: đi sang PHẢI trước")
        dfs(node.right, depth + 1)
        old = node.val
        total += old
        node.val = total
        print(f"{pad}thăm nút {old}: total = {total} -> ghi node.val = {total}")
        print(f"{pad}nút {old}: đi sang TRÁI")
        dfs(node.left, depth + 1)

    dfs(root, 0)
    return root

r = trace_recursive(build_tree([5, 2, 13]))
print("kết quả:", tree_to_list(r))
assert tree_to_list(r) == [18, 20, 13]

vào nút 5: đi sang PHẢI trước
    vào nút 13: đi sang PHẢI trước
        (None) -> quay lại
    thăm nút 13: total = 13 -> ghi node.val = 13
    nút 13: đi sang TRÁI
        (None) -> quay lại
thăm nút 5: total = 18 -> ghi node.val = 18
nút 5: đi sang TRÁI
    vào nút 2: đi sang PHẢI trước
        (None) -> quay lại
    thăm nút 2: total = 20 -> ghi node.val = 20
    nút 2: đi sang TRÁI
        (None) -> quay lại
kết quả: [18, 20, 13]


Quan sát: nút `13` (lớn nhất) được thăm đầu tiên nên `total = 13`; tiếp đó `5` cộng thành `18`; cuối cùng `2` cộng thành `20`. Thứ tự thăm đúng là **giảm dần** `13, 5, 2`.

### 8.2. Trace cách 4 (ngăn xếp)

Bảng: mỗi dòng ứng với **một lần pop**.

| Cột | Ý nghĩa |
|---|---|
| `đẩy vào` | các nút vừa được push khi đi sâu về phải |
| `stack` | nội dung ngăn xếp (đáy → đỉnh) sau khi push |
| `pop` | nút lấy ra xử lý |
| `total` | tổng chạy sau khi xử lý |

In [19]:
def trace_stack(root):
    stack, node, total = [], root, 0
    step = 0
    while stack or node:
        pushed = []
        while node:
            stack.append(node)
            pushed.append(node.val)
            node = node.right
        print(f"đẩy vào {pushed} | stack = {[n.val for n in stack]}")
        node = stack.pop()
        old = node.val
        total += old
        node.val = total
        step += 1
        print(f"  bước {step}: pop {old} -> total = {total}, node.val = {total} | stack = {[n.val for n in stack]}")
        node = node.left
        print(f"  rẽ trái -> node = {node.val if node else None}")
    return root

r = trace_stack(build_tree([4, 1, 6, 0, 2, 5, 7, None, None, None, 3, None, None, None, 8]))
print("kết quả:", tree_to_list(r))
assert tree_to_list(r) == examples[0][1]

đẩy vào [4, 6, 7, 8] | stack = [4, 6, 7, 8]
  bước 1: pop 8 -> total = 8, node.val = 8 | stack = [4, 6, 7]
  rẽ trái -> node = None
đẩy vào [] | stack = [4, 6, 7]
  bước 2: pop 7 -> total = 15, node.val = 15 | stack = [4, 6]
  rẽ trái -> node = None
đẩy vào [] | stack = [4, 6]
  bước 3: pop 6 -> total = 21, node.val = 21 | stack = [4]
  rẽ trái -> node = 5
đẩy vào [5] | stack = [4, 5]
  bước 4: pop 5 -> total = 26, node.val = 26 | stack = [4]
  rẽ trái -> node = None
đẩy vào [] | stack = [4]
  bước 5: pop 4 -> total = 30, node.val = 30 | stack = []
  rẽ trái -> node = 1
đẩy vào [1, 2, 3] | stack = [1, 2, 3]
  bước 6: pop 3 -> total = 33, node.val = 33 | stack = [1, 2]
  rẽ trái -> node = None
đẩy vào [] | stack = [1, 2]
  bước 7: pop 2 -> total = 35, node.val = 35 | stack = [1]
  rẽ trái -> node = None
đẩy vào [] | stack = [1]
  bước 8: pop 1 -> total = 36, node.val = 36 | stack = []
  rẽ trái -> node = 0
đẩy vào [0] | stack = [0]
  bước 9: pop 0 -> total = 36, node.val = 36 | stack = 

Cây ví dụ 1 có các lần pop theo thứ tự `8, 7, 6, 5, 4, 3, 2, 1, 0` (đúng **giảm dần**) và `total` lần lượt là `8, 15, 21, 26, 30, 33, 35, 36, 36`, khớp bảng tính tay ở mục 4.

### 8.3. Trace cách 5 (Morris)

Ta in từng lần `node` thay đổi, loại hành động (nối dây / gỡ dây / thăm) và các dây tạm đang tồn tại.

In [20]:
def trace_morris(root):
    total, node = 0, root
    step = 0
    while node:
        step += 1
        if node.right is None:
            old = node.val
            total += old
            node.val = total
            print(f"{step:>2}. node={old}: không có con phải -> THĂM, total={total}")
            node = node.left
        else:
            succ = node.right
            while succ.left and succ.left is not node:
                succ = succ.left
            if succ.left is None:
                succ.left = node
                print(f"{step:>2}. node={node.val}: succ={succ.val}, NỐI dây {succ.val}.left -> {node.val}, đi sang phải")
                node = node.right
            else:
                succ.left = None
                old = node.val
                total += old
                node.val = total
                print(f"{step:>2}. node={old}: succ={succ.val} đã nối -> GỠ dây, THĂM, total={total}")
                node = node.left
    return root

r = trace_morris(build_tree([5, 2, 13]))
print("kết quả:", tree_to_list(r))
assert tree_to_list(r) == [18, 20, 13]

 1. node=5: succ=13, NỐI dây 13.left -> 5, đi sang phải
 2. node=13: không có con phải -> THĂM, total=13
 3. node=5: succ=13 đã nối -> GỠ dây, THĂM, total=18
 4. node=2: không có con phải -> THĂM, total=20
kết quả: [18, 20, 13]


Đọc trace: nút `5` có con phải `13`, `succ` (nhỏ nhất cây phải) chính là `13`; nối `13.left -> 5` rồi sang `13`. Nút `13` không có con phải nên được **thăm trước** (`total = 13`) rồi đi `left` — nhờ dây tạm, ta nhảy ngược về `5`. Lần này `succ.left is node` nên gỡ dây, thăm `5` (`total = 18`), rồi sang `2` (`total = 20`).

### 8.4. Trace cách 2 (hai pha) trên ví dụ 1

Kết quả của hai pha khi `verbose=True`:

In [21]:
_ = convert_two_pass(build_tree(examples[0][0]), verbose=True)

khóa theo inorder: [0, 1, 2, 3, 4, 5, 6, 7, 8]
sau pha 2        : [36, 36, 35, 33, 30, 26, 21, 15, 8]


## 9. Code nộp LeetCode

Chọn **cách 4 hoặc cách 3** tùy sở thích. Ở đây chọn **cách 3 (đệ quy phải → gốc → trái)** vì ngắn nhất và đúng với giới hạn của đề (`n <= 10^4`); lưu ý cây lệch 10^4 nút có thể chạm giới hạn đệ quy mặc định của Python, nếu bạn muốn an toàn tuyệt đối hãy dùng cách 4 (ngăn xếp). Dưới đây là code đúng class/method/signature (trên LeetCode `TreeNode`, `Optional` có sẵn; ở đây ta đã định nghĩa/import ở trên):

In [22]:
class Solution:
    def convertBST(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        total = 0

        def dfs(node: Optional[TreeNode]) -> None:
            nonlocal total
            if node is None:
                return
            dfs(node.right)          # các khóa lớn hơn
            total += node.val        # cộng nút hiện tại vào tổng chạy
            node.val = total         # ghi đè bằng tổng chạy
            dfs(node.left)           # các khóa nhỏ hơn

        dfs(root)
        return root


for vals, expected in examples:
    got = tree_to_list(Solution().convertBST(build_tree(vals)))
    print(vals, "->", got)
    assert got == expected

[4, 1, 6, 0, 2, 5, 7, None, None, None, 3, None, None, None, 8] -> [30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8]
[0, None, 1] -> [1, None, 1]
[] -> []


## 10. Vì sao thuật toán đúng

**Bất biến của duyệt đảo.** Gọi `x1 > x2 > ... > xn` là các khóa của BST theo thứ tự giảm dần. Duyệt **phải → gốc → trái** thăm các nút đúng theo thứ tự `x1, x2, ..., xn`. Lý do (quy nạp theo chiều cao cây): tại nút `v`, mọi khóa ở cây phải đều `> v` nên đứng trước `v` trong thứ tự giảm; mọi khóa ở cây trái đều `< v` nên đứng sau. Duyệt đệ quy cây phải cho các khóa đó theo thứ tự giảm (giả thiết quy nạp), sau đó `v`, sau đó cây trái.

**Giá trị ghi vào nút.** Ngay trước khi thăm `xi`, `total = x1 + x2 + ... + x(i-1)` (đã cộng mọi khóa lớn hơn). Sau `total += xi` ta có `total = x1 + ... + xi` = **`xi` cộng với tổng mọi khóa lớn hơn** — đúng điều đề yêu cầu. Vì mọi khóa duy nhất nên "lớn hơn" không bị trùng lặp.

**Ghi đè an toàn.** Khóa gốc của `xi` chỉ được dùng **một lần** (khi `total += xi`), sau đó nút không bao giờ bị đọc lại bởi thuật toán, nên ghi đè không làm sai các nút sau.

**Cây rỗng.** `dfs(None)` không làm gì, trả về `None` — khớp đáp án `[]`.

**Các cách khác.** Cách 2 tính cùng tổng hậu tố trên list đã sắp (bằng inorder). Cách 4 là đúng thứ tự thăm của đệ quy. Cách 5 (Morris) thăm các nút theo cùng thứ tự đảo, vì dây tạm `succ.left = node` đảm bảo ta quay lại `node` **đúng sau khi** cây phải đã duyệt xong.

**Cách 1 (brute force)** đúng trực tiếp theo định nghĩa của đề, nên ta dùng nó làm "chuẩn" khi đối chiếu ở mục 12.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là chiều cao cây (`log n <= h <= n`).

| Cách | Time | Space (phụ) | Ghi chú |
|---|---|---|---|
| 1. Brute force | O(n²) | O(n) | quét lại cả cây cho mỗi nút |
| 2. Inorder ra list + tổng hậu tố | O(n) | O(n) | tách rõ hai pha |
| 3. Đệ quy đảo + tổng chạy | O(n) | O(h) | ngắn gọn nhất |
| 4. Đảo bằng stack | O(n) | O(h) | tránh `RecursionError` |
| 5. Morris đảo | O(n) | O(1) | khó đọc, tạm đổi cấu trúc cây |

**Chi tiết cách 3.** Mỗi nút được thăm đúng một lần (`n` lời gọi có nút thật, `n + 1` lời gọi trả về ngay vì `None`), mỗi lần làm `O(1)` việc nên `O(n)`. Chiều sâu đệ quy là `h`.

**Chi tiết cách 5.** Vòng `while succ.left ...` có vẻ lồng nhau nhưng mỗi cạnh "trái" của cây con phải chỉ bị đi qua tối đa **hai lần** trong cả thuật toán (một lần khi nối dây, một lần khi kiểm tra và gỡ), nên tổng vẫn `O(n)`.

Với `n = 10^4`: cách tuyến tính xử lý trong vài ms, cách 1 cần khoảng `10^8` phép toán (vài chục giây trong Python).

## 12. Bộ test

Ta kiểm tra theo nhiều lớp:

1. Ví dụ của đề cho cả 5 cách.
2. Edge case: cây rỗng, một nút, hai nút, toàn số âm, lệch trái/phải, giá trị biên.
3. **Phản ví dụ** của các ý tưởng sai.
4. Đối chiếu 5 cách với một **oracle độc lập** chỉ làm việc trên **list kiểu LeetCode** (không dùng `TreeNode`) trên cây ngẫu nhiên có seed.
5. Test lớn và so sánh thời gian.

In [23]:
def oracle_list(lst):
    """Oracle độc lập: làm việc trực tiếp trên list kiểu LeetCode.
    Giá trị mới của mỗi phần tử v = tổng mọi phần tử u (không None) với u >= v."""
    vals = [v for v in lst if v is not None]
    return [None if v is None else sum(u for u in vals if u >= v) for v in lst]


def oracle_fast(lst):
    """Oracle nhanh hơn (sắp xếp + tổng hậu tố) cho cây lớn."""
    vals = sorted(v for v in lst if v is not None)
    suffix, run = {}, 0
    for v in reversed(vals):
        run += v
        suffix[v] = run
    return [None if v is None else suffix[v] for v in lst]


for vals, expected in examples:
    assert oracle_list(vals) == expected == oracle_fast(vals)
print("oracle khớp ví dụ đề")

solvers = {
    "brute": lambda r: convert_brute(r),
    "two_pass": lambda r: Solution2().convertBST(r),
    "recursive": lambda r: Solution3().convertBST(r),
    "stack": lambda r: Solution4().convertBST(r),
    "morris": lambda r: Solution5().convertBST(r),
    "final": lambda r: Solution().convertBST(r),
}
for name, fn in solvers.items():
    for vals, expected in examples:
        assert tree_to_list(fn(build_tree(vals))) == expected, (name, vals)
print("tất cả cách khớp ví dụ đề:", list(solvers))

oracle khớp ví dụ đề
tất cả cách khớp ví dụ đề: ['brute', 'two_pass', 'recursive', 'stack', 'morris', 'final']


In [24]:
# --- Edge case ---
edge_cases = [
    [],                                   # cây rỗng
    [7],                                  # một nút
    [5, 3],                               # hai nút, con trái
    [5, None, 9],                         # hai nút, con phải
    [-3, -10, 4],                         # có số âm
    [-5, -10, -1],                        # toàn âm
    [0],                                  # chỉ có 0
    [3, 2, None, 1],                      # lệch trái
    [1, None, 2, None, 3],                # lệch phải
    [10000, -10000],                      # biên giá trị
    [-10000, None, 10000],
]
for vals in edge_cases:
    want = oracle_list(vals)
    for name, fn in solvers.items():
        got = tree_to_list(fn(build_tree(vals)))
        assert got == want, (name, vals, got, want)
    print(vals, "->", want)

[] -> []
[7] -> [7]
[5, 3] -> [5, 8]
[5, None, 9] -> [14, None, 9]
[-3, -10, 4] -> [1, -9, 4]
[-5, -10, -1] -> [-6, -16, -1]
[0] -> [0]
[3, 2, None, 1] -> [3, 5, None, 6]
[1, None, 2, None, 3] -> [6, None, 5, None, 3]
[10000, -10000] -> [10000, 0]
[-10000, None, 10000] -> [0, None, 10000]


**Phản ví dụ.** Hai ý tưởng "nghe có vẻ đúng" nhưng sai, ta chứng minh bằng một test cụ thể:

- *Ý tưởng sai A:* "mỗi nút cộng thêm tổng cây con **phải** gốc" (bỏ qua các khóa lớn hơn nằm ở **tổ tiên** và nhánh khác).
- *Ý tưởng sai B:* duyệt trung thứ tự thường (trái → gốc → phải) rồi tích lũy — ra tổng các khóa **nhỏ hơn hoặc bằng**.

In [25]:
def subtree_sum(node):
    return 0 if node is None else node.val + subtree_sum(node.left) + subtree_sum(node.right)

def wrong_right_subtree(root):
    def dfs(node):
        if node is None:
            return
        node.val += subtree_sum(node.right)     # chỉ cộng cây phải
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return root

def wrong_forward(root):
    total = 0
    def dfs(node):
        nonlocal total
        if node is None:
            return
        dfs(node.left)                          # SAI chiều: trái trước
        total += node.val
        node.val = total
        dfs(node.right)
    dfs(root)
    return root

vals = [5, 2, 13]
print("đúng        :", oracle_list(vals))
print("sai A       :", tree_to_list(wrong_right_subtree(build_tree(vals))))
print("sai B       :", tree_to_list(wrong_forward(build_tree(vals))))
vals = examples[0][0]
print("ví dụ 1, sai A:", tree_to_list(wrong_right_subtree(build_tree(vals))))
assert tree_to_list(wrong_right_subtree(build_tree(vals))) != examples[0][1]
assert tree_to_list(wrong_forward(build_tree(vals))) != examples[0][1]

đúng        : [18, 20, 13]
sai A       : [18, 2, 13]
sai B       : [7, 2, 20]
ví dụ 1, sai A: [30, 6, 21, 0, 5, 5, 15, None, None, None, 3, None, None, None, 8]


Với `[5,2,13]`: ý tưởng A cho nút `2` giá trị `2` (không cộng `5` và `13` vì chúng không nằm trong cây con phải của `2`), trong khi đáp án đúng là `20`. Ý tưởng B cho `[7, 2, 20]` (theo thứ tự gốc, trái, phải; đó là tổng các khóa `<=`: `5 -> 2+5`, `2 -> 2`, `13 -> 20`), khác hẳn.

**Đối chiếu nhiều input ngẫu nhiên (có seed, đúng ràng buộc).** Ta sinh BST bằng cách chọn `n` giá trị **phân biệt** trong `[-10^4, 10^4]` rồi **chèn theo thứ tự ngẫu nhiên** vào BST (nên hình dạng cây cũng ngẫu nhiên).

In [26]:
def random_bst_list(rng, n, lo=-10**4, hi=10**4):
    """BST ngẫu nhiên n nút, khóa phân biệt trong [lo, hi], trả về list kiểu LeetCode."""
    if n == 0:
        return []
    keys = rng.sample(range(lo, hi + 1), n)       # khóa phân biệt, thứ tự chèn ngẫu nhiên
    root = TreeNode(keys[0])
    for k in keys[1:]:
        cur = root
        while True:
            if k < cur.val:
                if cur.left is None:
                    cur.left = TreeNode(k)
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TreeNode(k)
                    break
                cur = cur.right
    return tree_to_list(root)


def is_bst(root, lo=float("-inf"), hi=float("inf")):
    if root is None:
        return True
    return lo < root.val < hi and is_bst(root.left, lo, root.val) and is_bst(root.right, root.val, hi)


def same_shape(a, b):
    """Hai cây có cùng hình dạng (bỏ qua giá trị) hay không, so sánh qua list."""
    return [x is None for x in a] == [x is None for x in b]


rng = random.Random(538)
count = 0
for _ in range(400):
    n = rng.randint(0, 60)
    vals = random_bst_list(rng, n)
    assert is_bst(build_tree(vals)) and len(vals) >= n
    want = oracle_list(vals)
    for name, fn in solvers.items():
        got = tree_to_list(fn(build_tree(vals)))
        assert got == want, (name, vals)
        assert same_shape(got, vals), (name, "hình dạng cây bị đổi")
    count += 1
print("đối chiếu", count, "cây ngẫu nhiên, 6 cách + oracle: OK")

đối chiếu 400 cây ngẫu nhiên, 6 cách + oracle: OK


In [27]:
# Cây lớn (n = 10^4) và cây suy biến: chỉ dùng cách không đệ quy sâu + final (có nâng giới hạn)
rng = random.Random(5380)
big = random_bst_list(rng, 10**4)
want_big = oracle_fast(big)
for name in ("stack", "morris"):
    assert tree_to_list(solvers[name](build_tree(big))) == want_big
print("n = 10^4 ngẫu nhiên: stack, morris khớp oracle_fast")

# cây lệch phải 10^4 nút: giá trị tăng dần chèn theo thứ tự -> một "cái que"
chain_vals = list(range(-5000, 5000))
chain = None
for k in reversed(chain_vals):                  # dựng trực tiếp: 4999 <- 4998 <- ...
    chain = TreeNode(k, None, chain)
chain_list = tree_to_list(chain)
want_chain = oracle_fast(chain_list)
for name in ("stack", "morris"):
    assert tree_to_list(solvers[name](build_tree(chain_list))) == want_chain
print("cây lệch phải 10^4 nút: stack, morris khớp")

# cây lệch trái 10^4 nút
left_chain = None
for k in chain_vals:
    left_chain = TreeNode(k, left_chain, None)
left_list = tree_to_list(left_chain)
for name in ("stack", "morris"):
    assert tree_to_list(solvers[name](build_tree(left_list))) == oracle_fast(left_list)
print("cây lệch trái 10^4 nút: stack, morris khớp")

n = 10^4 ngẫu nhiên: stack, morris khớp oracle_fast


cây lệch phải 10^4 nút: stack, morris khớp


cây lệch trái 10^4 nút: stack, morris khớp


In [28]:
# So sánh thời gian: brute O(n^2) vs các cách O(n)
rng = random.Random(77)
n = 1500
vals = random_bst_list(rng, n)
for name in ("brute", "two_pass", "recursive", "stack", "morris"):
    t0 = time.perf_counter()
    solvers[name](build_tree(vals))
    dt = time.perf_counter() - t0
    print(f"{name:>10}: {dt * 1000:8.2f} ms (n = {n}, gồm cả thời gian dựng cây)")

     brute:   119.41 ms (n = 1500, gồm cả thời gian dựng cây)
  two_pass:     0.66 ms (n = 1500, gồm cả thời gian dựng cây)
 recursive:     1.03 ms (n = 1500, gồm cả thời gian dựng cây)
     stack:     0.66 ms (n = 1500, gồm cả thời gian dựng cây)
    morris:     0.69 ms (n = 1500, gồm cả thời gian dựng cây)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: truyền tổng theo kiểu **tham trị** (không giữ lại khi quay lui)

Viết `dfs(node, s)` rồi `s += node.val` bên trong: số nguyên là bất biến nên `s` của lời gọi cha **không** nhận phần cộng từ cây con phải.

In [29]:
def wrong_by_value(root):
    def dfs(node, s):
        if node is None:
            return
        dfs(node.right, s)
        s += node.val               # chỉ đổi s cục bộ, cha không thấy
        node.val = s
        dfs(node.left, s)
    dfs(root, 0)
    return root

print("sai  :", tree_to_list(wrong_by_value(build_tree([5, 2, 13]))))
print("đúng :", tree_to_list(Solution().convertBST(build_tree([5, 2, 13]))))

sai  : [5, 7, 13]
đúng : [18, 20, 13]


**Vì sao sai.** Nút `5` gọi `dfs(13, 0)`; trong đó `s` cục bộ thành `13` nhưng **`s` của `5` vẫn là `0`**, nên `5` chỉ nhận `5` thay vì `18`. **Cách sửa:** dùng `nonlocal total` (hoặc `self.total`), hoặc cho `dfs` **trả về** tổng mới (`return s`) và gán lại `s = dfs(node.right, s)`.

Phiên bản sửa theo hướng trả về giá trị:

In [30]:
def fixed_by_return(root):
    def dfs(node, s):
        if node is None:
            return s
        s = dfs(node.right, s)       # nhận lại tổng sau khi xử lý cây phải
        s += node.val
        node.val = s
        return dfs(node.left, s)
    dfs(root, 0)
    return root

assert tree_to_list(fixed_by_return(build_tree([5, 2, 13]))) == [18, 20, 13]
assert tree_to_list(fixed_by_return(build_tree(examples[0][0]))) == examples[0][1]
print("OK")

OK


### Lỗi 2: duyệt sai chiều (trái → gốc → phải)

Duyệt inorder bình thường rồi tích lũy cho ra tổng các khóa **nhỏ hơn hoặc bằng** (một "Lesser Tree"), không phải Greater Tree. Mẹo nhớ: muốn "lớn hơn" thì phải đi **từ phía lớn**, tức là **phải trước**.

In [31]:
print("sai  :", tree_to_list(wrong_forward(build_tree([5, 2, 13]))))      # [7, 2, 20] = tổng các khóa <=
print("đúng :", tree_to_list(Solution().convertBST(build_tree([5, 2, 13]))))

sai  : [7, 2, 20]
đúng : [18, 20, 13]


### Lỗi 3: brute force **ghi giá trị mới ngay** trong lúc quét

Nếu vừa tính xong cho nút `x` là gán luôn `x.val`, các nút sau sẽ cộng nhầm giá trị **đã bị đổi**.

In [32]:
def wrong_brute_inplace(root):
    nodes = collect_nodes(root)
    for x in nodes:
        x.val = sum(m.val for m in nodes if m.val >= x.val)    # SAI: m.val có thể đã đổi
    return root

print("sai  :", tree_to_list(wrong_brute_inplace(build_tree([0, None, 1]))))
print("đúng :", tree_to_list(convert_brute(build_tree([0, None, 1]))))

sai  : [1, None, 2]
đúng : [1, None, 1]


**Vì sao sai.** Preorder cho `[0, 1]`. Nút `0` được đổi thành `0 + 1 = 1`. Khi tới nút `1`, điều kiện `m.val >= 1` giờ đúng với **cả `0` (nay là `1`) và `1`**, nên tổng thành `2`, trong khi đáp án đúng là `1`. **Cách sửa:** tính hết giá trị mới vào một list tạm, rồi mới gán (như `convert_brute`).

### Lỗi 4: biến đếm ở **mức class** dính giữa các lần gọi

Đặt `total = 0` ở thân class (không phải trong hàm) là biến **dùng chung cho mọi đối tượng và mọi lần gọi**, nên lần gọi thứ hai bắt đầu với tổng cũ.

In [33]:
class WrongSolution:
    total = 0                                   # SAI: biến mức class

    def convertBST(self, root):
        if root is not None:
            self.convertBST(root.right)
            WrongSolution.total += root.val
            root.val = WrongSolution.total
            self.convertBST(root.left)
        return root

print("lần 1:", tree_to_list(WrongSolution().convertBST(build_tree([5, 2, 13]))))
print("lần 2:", tree_to_list(WrongSolution().convertBST(build_tree([5, 2, 13]))))   # cộng dồn từ lần 1!
print("đúng :", tree_to_list(Solution().convertBST(build_tree([5, 2, 13]))))

lần 1: [18, 20, 13]
lần 2: [38, 40, 33]
đúng : [18, 20, 13]


**Vì sao sai.** Sau lần gọi 1, `WrongSolution.total` đã là `20` (= 13 + 5 + 2) và **không bị reset**; lần gọi 2 bắt đầu cộng từ `20` nên ra `[38, 40, 33]` thay vì `[18, 20, 13]`. Trên LeetCode, nhiều test case có thể dùng chung một lớp nên lỗi này hiện ra ở test thứ hai. **Cách sửa:** đặt `total = 0` **bên trong** `convertBST` (như bản nộp dùng `nonlocal`), hoặc gán `self.total = 0` ở đầu hàm **bao ngoài** (không phải trong hàm đệ quy, nếu không nó bị reset ở mỗi lời gọi đệ quy).

### Lỗi 5: đệ quy quá sâu với cây lệch

Cây lệch có chiều cao `n` (tới `10^4`) nhưng giới hạn đệ quy mặc định của Python là khoảng `1000`. Bản đệ quy sẽ gặp `RecursionError` với cây lệch đủ dài.

In [34]:
chain = None
for k in reversed(range(3000)):               # chuỗi lệch phải 3000 nút: 0 -> 1 -> ... -> 2999
    chain = TreeNode(k, None, chain)
chain_list = tree_to_list(chain)

print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
try:
    Solution().convertBST(build_tree(chain_list))
    print("không lỗi (môi trường cho phép sâu hơn)")
except RecursionError:
    print("RecursionError với cây lệch 3000 nút")

# Cách 1 để sửa: dùng bản lặp (stack hoặc Morris)
got = tree_to_list(Solution4().convertBST(build_tree(chain_list)))
assert got == oracle_fast(chain_list)
print("bản stack chạy ổn")

# Cách 2: nâng giới hạn tạm thời (rồi trả về giá trị cũ)
old = sys.getrecursionlimit()
sys.setrecursionlimit(20000)
got = tree_to_list(Solution().convertBST(build_tree(chain_list)))
sys.setrecursionlimit(old)
assert got == oracle_fast(chain_list)
print("nâng giới hạn đệ quy rồi chạy bản đệ quy: OK")

giới hạn đệ quy mặc định: 1000
RecursionError với cây lệch 3000 nút
bản stack chạy ổn
nâng giới hạn đệ quy rồi chạy bản đệ quy: OK


**Cách sửa.** Dùng bản lặp (cách 4 hoặc 5), hoặc `sys.setrecursionlimit(...)` trước khi chạy. Trên LeetCode, bản đệ quy thường vẫn qua vì hệ thống chấm đã nâng giới hạn, nhưng khi tự chạy tại máy hãy nhớ điều này.

### Lỗi 6: nhầm "lớn hơn" với "lớn hơn hoặc bằng" và quên cộng chính nút

Đề nói "khóa gốc **cộng** tổng các khóa lớn hơn", tức **tính cả chính nút**. Một số người cộng `total` *trước*, ghi vào nút *rồi mới* cộng `node.val`, kết quả thiếu mất giá trị của chính nút.

In [35]:
def wrong_order_of_ops(root):
    total = 0
    def dfs(node):
        nonlocal total
        if node is None:
            return
        dfs(node.right)
        node.val = total            # SAI: ghi trước khi cộng chính nó
        total += node.val
        dfs(node.left)
    dfs(root)
    return root

print("sai  :", tree_to_list(wrong_order_of_ops(build_tree([5, 2, 13]))))     # thiếu chính nó
print("đúng :", tree_to_list(Solution().convertBST(build_tree([5, 2, 13]))))

sai  : [0, 0, 0]
đúng : [18, 20, 13]


**Vì sao sai.** Dòng `node.val = total` ghi tổng của các khóa **lớn hơn** (không có chính nó), rồi `total += node.val` cộng **giá trị đã bị ghi đè** (không phải khóa gốc) nên tổng chạy cũng hỏng. Thứ tự đúng: `total += node.val` rồi `node.val = total`.

## 14. Mẹo & nhận dạng pattern

- **"Mỗi nút += tổng các nút lớn/nhỏ hơn" trên BST** → nghĩ ngay tới **duyệt đảo/duyệt thuận + tổng chạy**. Dấu hiệu: đề nhắc BST và một đại lượng tích lũy theo thứ tự khóa.
- **Inorder của BST là dãy tăng.** Mọi bài BST có thể quy về "bài trên mảng đã sắp" bằng cách inorder; khi cần hiệu quả hơn, duyệt trực tiếp và giữ trạng thái chạy (như tổng chạy ở đây).
- **Muốn lớn → nhỏ thì đảo thứ tự con**: trái ↔ phải. Không cần sort.
- **Trạng thái dùng chung giữa các lời gọi đệ quy** → `nonlocal`/`self.x` (khởi tạo bên trong hàm gọi ngoài cùng) hoặc **trả giá trị về**. Không bao giờ dùng biến mức class cho trạng thái của một lần gọi.
- **Sửa tại chỗ** gọn, nhưng nhớ: test phải dựng cây mới cho mỗi lần chạy.
- **Chọn cách theo ràng buộc:** `n <= 10^4` cho phép đệ quy nếu nâng giới hạn; muốn chắc ăn dùng stack; Morris khi đề đòi `O(1)` bộ nhớ.
- Bài **1038** là bản y hệt (tên khác); các bài **230 (Kth Smallest in BST)**, **653 (Two Sum IV – BST)**, **98 (Validate BST)** cùng dựa vào tính chất inorder.

## 15. Biến thể

### 15.1. Lesser Tree: mỗi nút = tổng các khóa **nhỏ hơn hoặc bằng**

Duyệt **thuận** (trái → gốc → phải) với tổng chạy.

In [36]:
def lesser_tree(root):
    total = 0
    def dfs(node):
        nonlocal total
        if node is None:
            return
        dfs(node.left)
        total += node.val
        node.val = total
        dfs(node.right)
    dfs(root)
    return root

print(tree_to_list(lesser_tree(build_tree([5, 2, 13]))))     # [7, 2, 20]
assert tree_to_list(lesser_tree(build_tree([5, 2, 13]))) == [7, 2, 20]

[7, 2, 20]


### 15.2. Greater Tree **nghiêm ngặt**: tổng các khóa **lớn hơn**, *không* cộng chính nó

Chỉ cần đổi thứ tự hai thao tác: ghi trước, cộng sau (nhưng cộng **khóa gốc**, nên phải lưu `old`).

In [37]:
def strictly_greater(root):
    total = 0
    def dfs(node):
        nonlocal total
        if node is None:
            return
        dfs(node.right)
        old = node.val
        node.val = total            # tổng các khóa lớn hơn (không gồm chính nó)
        total += old
        dfs(node.left)
    dfs(root)
    return root

got = tree_to_list(strictly_greater(build_tree([5, 2, 13])))
print(got)                                   # [13, 18, 0]
assert got == [13, 18, 0]

[13, 18, 0]


### 15.3. Phần tử lớn thứ k trong BST (liên quan bài 230)

Duyệt đảo và dừng ở lần thăm thứ `k`.

In [38]:
def kth_largest(root, k):
    stack, node = [], root
    while stack or node:
        while node:
            stack.append(node)
            node = node.right
        node = stack.pop()
        k -= 1
        if k == 0:
            return node.val
        node = node.left

tree = build_tree(examples[0][0])
print([kth_largest(tree, k) for k in range(1, 10)])         # 8, 7, ..., 0
assert [kth_largest(tree, k) for k in range(1, 10)] == [8, 7, 6, 5, 4, 3, 2, 1, 0]

[8, 7, 6, 5, 4, 3, 2, 1, 0]


### 15.4. Tổng các khóa trong đoạn `[low, high]` (liên quan bài 938)

Tận dụng tính chất BST để **cắt nhánh**: nếu `node.val < low` thì chỉ xét cây phải; nếu `> high` thì chỉ xét cây trái.

In [39]:
def range_sum_bst(root, low, high):
    if root is None:
        return 0
    if root.val < low:
        return range_sum_bst(root.right, low, high)
    if root.val > high:
        return range_sum_bst(root.left, low, high)
    return root.val + range_sum_bst(root.left, low, high) + range_sum_bst(root.right, low, high)

tree = build_tree(examples[0][0])
print(range_sum_bst(tree, 2, 5))                             # 2+3+4+5 = 14
assert range_sum_bst(tree, 2, 5) == 14

14


### 15.5. Biến thể khác để tự nghĩ

- Cho **mảng đã sắp** thay vì cây: đáp án chỉ là tổng hậu tố (mục 3.5).
- **Cây nhị phân thường** (không phải BST): không còn thứ tự miễn phí; phải gom khóa, sắp xếp, rồi tra bảng (như bước kiểm chứng ở mục 4) với `O(n log n)`.
- **Khóa trùng nhau**: phải định nghĩa lại "lớn hơn" (nghiêm ngặt hay không), và duyệt đảo cần xử lý nhóm khóa bằng nhau.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | mỗi nút quét cả cây cộng các khóa `>=` | O(n²) | O(n) | chỉ để kiểm chứng/oracle, `n` nhỏ |
| 2. Inorder ra list + hậu tố | sắp theo inorder rồi tổng hậu tố, gán lại | O(n) | O(n) | cần hiểu rõ hai pha; dễ gỡ lỗi |
| 3. Đệ quy đảo + tổng chạy | phải → gốc → trái, `nonlocal total` | O(n) | O(h) | ngắn nhất, dùng khi nộp bài/phỏng vấn |
| 4. Đảo bằng stack | mô phỏng đệ quy bằng `list` | O(n) | O(h) | cây có thể rất lệch, tránh `RecursionError` |
| 5. Morris đảo | dây tạm `succ.left = node` | O(n) | O(1) | đề đòi `O(1)` bộ nhớ phụ |

**Lời khuyên.** Học cách 3 trước (cốt lõi), nắm cách 4 (an toàn), biết cách 5 như kiến thức mở rộng. Cách 1 và 2 là bước đệm giúp bạn hiểu *vì sao* tổng chạy đúng.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [40]:
print(tree_to_list(Solution().convertBST(build_tree([5, 2, 13]))))
print(tree_to_list(Solution().convertBST(build_tree([0, None, 1]))))
print(tree_to_list(Solution().convertBST(build_tree([]))))

[18, 20, 13]
[1, None, 1]
[]


**Bài 2 (tính tay).** Cây `[2,1,3]` (gốc `2`, con trái `1`, con phải `3`). Tính Greater Tree rồi gán list kết quả (theo thứ tự LeetCode) vào `ans2` (đang là `None`).

In [41]:
ans2 = None      # TODO: list 3 số nguyên, ví dụ [?, ?, ?]

**Bài 3 (điền TODO).** Viết `suffix_sums(nums)`: nhận list **đã sắp tăng**, trả về list tổng hậu tố, ví dụ `[1,2,3] -> [6,5,3]`. Đây chính là pha 2 của cách 2.

In [42]:
def suffix_sums(nums):
    # TODO: đi từ phải sang trái với tổng chạy
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_convert(root)` bằng duyệt đảo **không đệ quy** (dùng stack), trả về `root`.

In [43]:
def my_convert(root):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `lesser_equal_tree(root)` cho biến thể "mỗi nút = tổng các khóa **nhỏ hơn hoặc bằng**" **không đệ quy**. (b) Viết `sum_greater_than(root, x)` trả về **tổng các khóa lớn hơn `x`** (với `x` bất kỳ, có thể không có trong cây) bằng cách tận dụng tính chất BST để **cắt nhánh** (bỏ qua nguyên cây trái khi gốc `<= x`).

In [44]:
def lesser_equal_tree(root):
    # TODO
    pass

def sum_greater_than(root, x):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: nút lớn nhất giữ nguyên. Sắp các khóa giảm dần rồi cộng dồn.
- Bài 2: khóa là `1,2,3`; đáp án của `3` là chính nó.
- Bài 3: khởi tạo `run = 0` và list kết quả cùng độ dài với `nums`.
- Bài 4: mẫu `while stack or node:` rồi đi hết bên phải.
- Bài 5: (a) đổi chiều duyệt; (b) nếu `root.val <= x` thì cả cây trái và gốc đều bị loại.

**Mức 2 (vừa):**
- Bài 2: `3 -> 3`, `2 -> 5`, `1 -> 6`; nhớ xếp theo thứ tự `[gốc, trái, phải]`.
- Bài 3: `for i in range(len(nums) - 1, -1, -1): run += nums[i]; out[i] = run`.
- Bài 4: sau khi pop: `total += node.val; node.val = total; node = node.left`.
- Bài 5: (a) dùng stack đi hết bên **trái**; (b) nếu `root.val > x` thì gốc và cả cây phải đều tính (cần tổng cây phải, dùng hàm phụ `subtree_sum`), rồi xét tiếp cây trái; ngược lại chỉ xét cây phải. (Với hàm phụ này, thời gian thực tế là `O(n)` ở trường hợp xấu nhất; mục tiêu của bài là luyện **cắt nhánh**.)

**Mức 3 (gần đáp án):**
- Bài 3: xem pha 2 của cách 2 (mục 7.1).
- Bài 4: xem cách 4, chỉ cần bỏ phần in.
- Bài 5: (a) xem cách 4 và đổi `right` ↔ `left`; (b) `if root.val <= x: return sum_greater_than(root.right, x)`, ngược lại `root.val + subtree_sum(root.right) + sum_greater_than(root.left, x)`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [45]:
# Bài 1
assert tree_to_list(Solution().convertBST(build_tree([5, 2, 13]))) == [18, 20, 13]
assert tree_to_list(Solution().convertBST(build_tree([0, None, 1]))) == [1, None, 1]
assert tree_to_list(Solution().convertBST(build_tree([]))) == []

# Bài 2: khóa 3 -> 3, 2 -> 5, 1 -> 6; list [gốc, trái, phải]
ans2 = [5, 6, 3]
assert ans2 == tree_to_list(Solution().convertBST(build_tree([2, 1, 3])))
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [46]:
# Bài 3
def suffix_sums(nums):
    out = [0] * len(nums)
    run = 0
    for i in range(len(nums) - 1, -1, -1):
        run += nums[i]
        out[i] = run
    return out

assert suffix_sums([1, 2, 3]) == [6, 5, 3]
assert suffix_sums([]) == []
assert suffix_sums([-4]) == [-4]
assert suffix_sums([0, 1, 2, 3, 4]) == [10, 10, 9, 7, 4]
print("Bài 3 OK")

Bài 3 OK


In [47]:
# Bài 4
def my_convert(root):
    stack, node, total = [], root, 0
    while stack or node:
        while node:
            stack.append(node)
            node = node.right
        node = stack.pop()
        total += node.val
        node.val = total
        node = node.left
    return root

rng = random.Random(4)
for _ in range(200):
    vals = random_bst_list(rng, rng.randint(0, 40))
    assert tree_to_list(my_convert(build_tree(vals))) == oracle_list(vals)
print("Bài 4 OK")

Bài 4 OK


In [48]:
# Bài 5 (a)
def lesser_equal_tree(root):
    stack, node, total = [], root, 0
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        total += node.val
        node.val = total
        node = node.right
    return root

assert tree_to_list(lesser_equal_tree(build_tree([5, 2, 13]))) == [7, 2, 20]
assert tree_to_list(lesser_equal_tree(build_tree([]))) == []
rng = random.Random(55)
for _ in range(200):
    vals = random_bst_list(rng, rng.randint(0, 40))
    keys = [v for v in vals if v is not None]
    want = [None if v is None else sum(u for u in keys if u <= v) for v in vals]
    assert tree_to_list(lesser_equal_tree(build_tree(vals))) == want

# Bài 5 (b)
def subtree_sum2(node):
    return 0 if node is None else node.val + subtree_sum2(node.left) + subtree_sum2(node.right)

def sum_greater_than(root, x):
    if root is None:
        return 0
    if root.val <= x:                                   # gốc và cây trái đều <= x: bỏ
        return sum_greater_than(root.right, x)
    return root.val + subtree_sum2(root.right) + sum_greater_than(root.left, x)

tree = build_tree(examples[0][0])
assert sum_greater_than(tree, 4) == 5 + 6 + 7 + 8
assert sum_greater_than(tree, 100) == 0
assert sum_greater_than(tree, -1) == sum(range(9))
assert sum_greater_than(None, 3) == 0
rng = random.Random(56)
for _ in range(200):
    vals = random_bst_list(rng, rng.randint(0, 40))
    keys = [v for v in vals if v is not None]
    x = rng.randint(-10**4, 10**4)
    assert sum_greater_than(build_tree(vals), x) == sum(u for u in keys if u > x)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đệ quy, phải → gốc → trái):**

```python
class Solution:
    def convertBST(self, root):
        total = 0

        def dfs(node):
            nonlocal total
            if node is None:
                return
            dfs(node.right)          # khóa lớn hơn
            total += node.val        # cộng chính nó
            node.val = total         # ghi đè
            dfs(node.left)           # khóa nhỏ hơn

        dfs(root)
        return root
```

**Template (lặp, stack):**

```python
class Solution:
    def convertBST(self, root):
        stack, node, total = [], root, 0
        while stack or node:
            while node:
                stack.append(node)
                node = node.right
            node = stack.pop()
            total += node.val
            node.val = total
            node = node.left
        return root
```

**Câu thần chú:** *BST + "lớn hơn" ⇒ đi phải trước; tổng chạy; cộng rồi mới ghi đè.*

**Checklist trước khi submit:**

- [ ] Duyệt theo thứ tự **phải → gốc → trái** (không phải trái → gốc → phải)?
- [ ] `total` là biến dùng chung (`nonlocal`/`self.`), **không** phải tham số truyền theo trị, và **không** là biến mức class?
- [ ] Thứ tự thao tác: `total += node.val` **trước**, `node.val = total` **sau**?
- [ ] `root is None` -> trả `None` (cây rỗng) mà không lỗi?
- [ ] Trả về `root` (gốc của cây đã đổi)?
- [ ] Cây lệch 10^4 nút: đã nghĩ tới `RecursionError` (dùng stack/Morris hoặc nâng giới hạn)?
- [ ] Đã thử: cây rỗng, một nút, toàn số âm, cây lệch trái/phải, biên `±10^4`, ví dụ 1 của đề?